# CHELSA-TraCE21k → yearly-mean netCDF

Converts CHELSA-TraCE21k monthly temperature GeoTIFFs (`tasmin` / `tasmax`)
into one netCDF per variable, holding the **yearly mean** at each
**100-year (centennial) time step**, at full ~1 km resolution.

## Configuration

- `VARIABLES` — which variables to process (e.g. `["tasmin"]`).
- `BASE_DIR` / `INPUT_DIR` / `OUTPUT_DIR` — input and output locations.
- `SCALE_FACTOR = 0.1` — CHELSA stores temperature as Kelvin × 10; this rescales it.
- `COARSEN_FACTOR` — `None` for full resolution, or an int for N×N block-mean downsampling.
- `LONG_NAMES` — per-variable metadata description, looked up by variable name.
- `CHUNKS = {"x": 4500, "y": 2250}` — chunk size for the streaming write.
- `dask.config.set(num_workers=4)` — caps parallel tasks.

## Helpers

- **`index_to_year(time_index)`**:
    - Maps a CHELSA time index to a calendar year.
    - Anchor `t = 20 → 1990 CE`, each step = 100 years.
    - Each filename looks like `CHELSA_TraCE21k_tasmax_<MM>_<TTT>_V.1.0.tif`, where `MM` is the month (`01`–`12`) and `TTT` is a CHELSA time index. The script reads the time index from the filename and converts it to a calendar year
    - Mapping:
    | TTT     | Year (CE) | Comment             |
    |---------|-----------|---------------------|
    | `0020`  | 1990      | most recent step    |
    | `0000`  | -10       | ~ year 0            |
    | `-001`  | -110      | 100 years older     |
    | `-200`  | -20010    | ~ 22 ka BP, ≈ LGM   |

    - **DISCAIMER**: 
    The mapping `year = 1990 − (20 − TTT) × 100` is a hypothesis. CHELSA's
    official documentation does not explicitly publish the index-to-year table.
    The hypothesis is consistent with:
        - the metadata field `frequency=centennial` in your `.tif` files,
        - the paper's claim of ~21,000 years of coverage,
        - the file count (221 timesteps × 100 years ≈ 22 ka).

- **`group_files_by_year(variable)`**:
    - globs the input folder, matches each filename against a regex to pull out the month and time index, converts the index to a year, and returns a dict mapping each year to its list of monthly file paths.

- **`lazy_yearly_mean(monthly_files)`**:
    - opens the 12 monthly rasters, optionally coarsens them, stacks along a `month` dimension and averages to one `(y, x)` field, applies the scale factor, and casts to `float32`.

## Main loop

For each variable:

1. Group its files by year and stop early if none are found.
2. For each year (in chronological order), check it has 12 months, build the
   yearly-mean field, and stamp it with a mid-year timestamp on a new `time` axis.
3. Concatenate all years along `time`, rename the spatial axes to `lat`/`lon`,
   and attach CF-style metadata (`long_name`, `units`, `source`, `comment`).
4. Write to netCDF — gzip-compressed, `float32`, with `time` as an unlimited
   (appendable) dimension.

## Laziness & memory

Every step before the write is **lazy**: `open_rasterio(chunks=...)`, `concat`,
`mean`, `* SCALE_FACTOR`, `astype`, and `expand_dims` only build a dask task
graph — no data is read or computed. Data is materialized **chunk by chunk**
only during `to_netcdf`, so peak RAM is set by the chunk size (≈ 40 MB per
chunk: 4500 × 2250 × 4 bytes) times the number of workers, not by the full
array. The `.astype("float32")` after scaling matters because multiplying by the
Python float `0.1` upcasts to float64; casting back keeps data at 4 bytes/pixel.

## Output

One netCDF per variable, dimensions `(time, lat, lon)`, self-describing via
metadata in the header.

In [ ]:
import re
from pathlib import Path
import cftime
import dask
import rioxarray
import xarray as xr
from dask.diagnostics import ProgressBar
from tqdm import tqdm
import subprocess
import tarfile
from tempfile import TemporaryDirectory

In [ ]:
def index_to_year(time_index):
    """Converts CHELSA centennial time index to calendar year. Anchor: t=20 -> 1990 CE."""
    return 1990 - (20 - time_index) * 100


def group_files_by_year(variable):
    """Maps each calendar year to its list of monthly GeoTIFF paths."""
    pattern = re.compile(rf"CHELSA_TraCE21k_{variable}_(\d{{2}})_(-?\d+)_V\.1\.0\.tif")
    files_by_year = {}
    for path in INPUT_DIR.glob(f"CHELSA_TraCE21k_{variable}_*.tif"):
        match = pattern.match(path.name)
        if match:
            year = index_to_year(int(match.group(2)))
            files_by_year.setdefault(year, []).append(path)
    return files_by_year


def lazy_yearly_mean(monthly_files):
    """Lazy 12-month mean for one year. Nothing is computed until the write."""
    months = []
    for path in monthly_files:
        da = rioxarray.open_rasterio(path, chunks=CHUNKS, masked=True).squeeze("band", drop=True)
        if COARSEN_FACTOR:
            da = da.coarsen(x=COARSEN_FACTOR, y=COARSEN_FACTOR, boundary="trim").mean()
        months.append(da)

    yearly_mean = xr.concat(months, dim="month").mean(dim="month")
    # *0.1 upcasts to float64; cast back so chunks and on-disk data stay 4 bytes/pixel.
    return (yearly_mean * SCALE_FACTOR).astype("float32")

In [ ]:
VARIABLES = [
    "tasmin",
    "tasmax"
]

BASE_DIR = Path.cwd() / "data" / "CHELSA-TraCE21k-centennial"
INPUT_DIR = Path.cwd() / "data" / "CHELSA-TraCE21k-centennial" / "data"
OUTPUT_DIR = Path.cwd() / "data" / "CHELSA-TraCE21k-centennial" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE_FACTOR = 0.1     # CHELSA stores temperature as Kelvin × 10
COARSEN_FACTOR = None  # set to an int for block-mean downsampling (lower resolution and smaller file size)

LONG_NAMES = {
    "tasmax": "Yearly mean of daily maximum near-surface air temperature",
    "tasmin": "Yearly mean of daily minimum near-surface air temperature",
}

# Chunk = unit of work for the streaming write. Each chunk is computed, written,
# then freed, so chunk size sets peak RAM (not the full array). 4500×2250 float32 ≈ 40 MB.
CHUNKS = {"x": 4500, "y": 2250}

# Cap parallel tasks so peak RAM stays bounded during the streaming write.
dask.config.set(num_workers=4)

In [ ]:
for variable in VARIABLES:
    print(f"Processing variable: {variable}")

    files_by_year = group_files_by_year(variable)
    if not files_by_year:
        raise SystemExit(f"No matching .tif files found in {INPUT_DIR}")

    slices = []
    for year in tqdm(sorted(files_by_year), desc="Building yearly means"):
        monthly_files = files_by_year[year]
        if len(monthly_files) != 12:
            raise ValueError(f"Year {year} has {len(monthly_files)} months, expected 12")

        timestamp = cftime.DatetimeProlepticGregorian(year, 7, 1) # builds a timestamp at July 1st of that year.
        slices.append(lazy_yearly_mean(monthly_files).expand_dims(time=[timestamp]))

    combined = xr.concat(slices, dim="time").rename({"x": "lon", "y": "lat"})
    combined.name = variable
    combined.attrs = {
        "long_name": LONG_NAMES[variable],
        "units": "K",
        "source": "CHELSA-TraCE21k v1.0 (centennial)",
        "comment": "Each timestep is a centennial climatology averaged over 12 months.",
    }
    if COARSEN_FACTOR:
        combined.attrs["spatial_processing"] = (f"Spatially downsampled by factor {COARSEN_FACTOR} (block-mean)")

    output_path = OUTPUT_DIR / f"{variable}.nc"
    print(f"Writing {output_path}")
    with ProgressBar():
        combined.to_netcdf(
            output_path,
            encoding={variable: {"zlib": True, "complevel": 4, "dtype": "float32"}},
            unlimited_dims=["time"],
        )
    print("Done.")

# Compute tasmean from tasmax and tasmin
- Using cdo
- (tasmax+tasmin)/2

In [ ]:
output_dir = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'output'
tasmin_nc  = output_dir / 'tasmin.nc'
tasmax_nc  = output_dir / 'tasmax.nc'
tasmean_nc = output_dir / 'tasmean.nc'

# Build CDO command
cmd = [
    'cdo',
    'expr,tasmean=(tasmin+tasmax)/2',
    '-merge', str(tasmin_nc), str(tasmax_nc),
    str(tasmean_nc),
]

# Run it and stream stdout/stderr into the notebook
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print(f'Exit code: {result.returncode}')
print(f'Output file: {tasmean_nc} (exists: {tasmean_nc.exists()})')

## PalMod2: Yearly Mean Computation & Merge

Computes annual means from monthly NetCDF files and merges them into a single output file using **CDO** (Climate Data Operators).

**Steps:**
1. Collect all `.nc` files from `INPUT_DIR`.
2. For each file: apply `cdo yearmean` (collapses 12 monthly values → 1 yearly mean) and save to a temporary directory.
3. Merge all yearly-mean files along the time axis with `cdo mergetime` into a single output file.

**Notes:**
- Temporary files are auto-deleted after the cell finishes.
- `tqdm` shows progress for the per-file step (the slow part).
- Time axis runs forward in model years (oldest → youngest), as the simulation integrates forward from the Last Glacial Maximum.

In [ ]:
import subprocess
import shutil
import tarfile
from pathlib import Path
from tempfile import TemporaryDirectory
from tqdm import tqdm

DATA_DIR = Path("/home/luser/Documents/Uni/TU/4 Semester/Geoinformatics Project/paleoclimate_model_comparison/data/PalMod2/data")
OUTPUT_DIR = DATA_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets = [
    "PMMXMCRTDGr111Amtasgn30201_1-250",
    "PMMXMCRTDGr111Lmtslgn30201_1-250",
]


def process_dataset(name):
    extract_dir = DATA_DIR / name
    output_nc = OUTPUT_DIR / f"{name}.nc"
    print(f"\n=== {name} ===")

    with tarfile.open(DATA_DIR / f"{name}.tar") as tar:
        tar.extractall(extract_dir, filter="data")

    nc_files = sorted(extract_dir.glob("**/*.nc"))
    print(f"Found {len(nc_files)} files.")

    with TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        yearmean_files = []
        for f in tqdm(nc_files, desc="Computing yearly means"):
            out = tmp / f"{f.stem}_ymean.nc"
            # cwd + relative name so CDO never sees the space-containing path
            subprocess.run(["cdo", "yearmean", str(f.relative_to(extract_dir)), str(out)],
                           check=True, cwd=extract_dir)
            yearmean_files.append(str(out))

        print("Merging...")
        merged = tmp / f"{name}.nc"  # merge in the temp dir (no spaces), then move
        subprocess.run(["cdo", "mergetime", *yearmean_files, str(merged)], check=True)
        shutil.move(str(merged), str(output_nc))

    print(f"Done: {output_nc}")


for name in datasets:
    process_dataset(name)